# 2008 WRR dry-bottom sloping-bed benchmark — AVAC

This notebook verifies AVAC's water limit against the analytical dry-front and rear-wave solution used in the original GeoClaw/TsunamiClaw benchmark. The one-dimensional problem is extruded across a five-cell periodic strip and the centerline is evaluated.


## Reproducible environment

The first code cell installs the repository's validation package and Python dependencies into the active kernel. A clean checkout also needs GNU Make and gfortran to compile the selected solver on first use.


In [ ]:
from pathlib import Path
SEARCH_ROOT = Path.cwd().resolve()
REPOSITORY = next(candidate for candidate in (SEARCH_ROOT, *SEARCH_ROOT.parents) if (candidate / 'validation' / 'pyproject.toml').is_file())
%pip install -q -e {REPOSITORY / 'validation'}


In [ ]:
import os
from avac4qgis_validation import validation_case
case = validation_case('AVAC', '2008_WRR_sloping_bed')
CORES = max(1, os.cpu_count() or 1)
case.path


## Physical and numerical definition

The bed is a frictionless 10° slope. The initial water depth is the exact triangular wedge $h=H_0(1-x/x_b)$ on $x_b\leq x\leq0$, with $H_0=1$ m and $x_b=-H_0/\tan(10°)$. The upstream boundary is a wall, the downstream boundary is extrapolating, and the transverse boundaries are periodic. AVAC is configured in its frictionless water mode; no avalanche rheology contributes to this run.


In [ ]:
DX_M = 0.005
T_FINAL_S = 5.0
OUTPUT_FRAMES = 100
case.run('run_avac_validation.py', '--dx', DX_M, '--t-final', T_FINAL_S, '--nout', OUTPUT_FRAMES, '--cores', CORES)


## Quantitative diagnostics

The summary reports front errors, rear-wave errors, initial-condition error, mass variation, grid spacing, core count, and the exact solver hash.


In [ ]:
summary = case.json('results/summary.json')
summary


## Comparison with theory

The marker curves are AVAC centerline measurements; continuous analytical curves are evaluated independently by the driver.


In [ ]:
case.show('figures/wrr_characteristics_avac_vs_theory.png', 'figures/wrr_depth_profiles_avac_vs_theory.png', 'figures/wrr_surface_profile_avac.png')
